In [3]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("../output")

titles = pd.read_csv(OUTPUT_DIR / "posts_with_topics.csv", usecols=["id", "title"])
segments_df = pd.read_csv(OUTPUT_DIR / "posts_segmented.csv").merge(titles, on="id", how="left")
ab_results_df = pd.read_csv(OUTPUT_DIR / "ab_test_results.csv")

In [5]:
#  segment별 추천 전략 작성
def generate_segment_recommendations(segments_df, min_posts=10):
    recommendations = []

    for segment in sorted(segments_df['segment'].unique()):
        seg_posts = segments_df[segments_df['segment'] == segment]

        topic_stats = seg_posts.groupby('topic')['score'].agg(['mean', 'count'])
        topic_stats = topic_stats[topic_stats['count'] >= min_posts]
        top_topic = topic_stats['mean'].idxmax()

        top_posts = seg_posts[seg_posts['topic'] == top_topic].sort_values('score', ascending=False).head(3)
        top_post_titles = top_posts['title'].tolist()

        recommendations.append({
            'segment': segment,
            'recommended_topic': top_topic,
            'top_posts': top_post_titles
        })

    return pd.DataFrame(recommendations)

segment_recommendations = generate_segment_recommendations(segments_df)
segment_recommendations

,segment,recommended_topic,top_posts
0,0,81,[Where can I find good datasets for market ris...
1,1,92,[South Korea releases 7382 COVID-19 case detai...
2,2,47,[Where can someone find a public dataset on wh...
3,3,46,[Looking for dataset for a stock prediction (p...


In [7]:
# AB 테스트 결과 병합
ab_cols = ab_results_df[['topic', 'diff', 'p_adj', 'significant']].rename(columns={'topic': 'recommended_topic'})
final_recommendations = segment_recommendations.merge(ab_cols, on='recommended_topic', how='left')

final_recommendations.to_csv(OUTPUT_DIR / "recommendations_final.csv", index=False)
final_recommendations

,segment,recommended_topic,top_posts,diff,p_adj,significant
0,0,81,[Where can I find good datasets for market ris...,NaN,NaN,NaN
1,1,92,[South Korea releases 7382 COVID-19 case detai...,NaN,NaN,NaN
2,2,47,[Where can someone find a public dataset on wh...,-0.037581,0.895862,False
3,3,46,[Looking for dataset for a stock prediction (p...,0.074917,0.735735,False
